# ANRF AISEHack 2.0 — Round 3: Polymer Property Prediction

Seven-target pipeline (Tg, Egc, Egb, Ei, Eea, EPS, Nc). Core pipeline (feature engineering,
true-partner features, physics blending, GBDT/NN/CNN ensemble, Ridge stacking) carried over
unchanged from the Round 2 notebook. Round 3 additions: auxiliary-data self-supervised
pretraining, polymer-invariance augmentation + consistency regularization, SHAP explainability.

**Compliance notes baked into this notebook (Official Rules §6):**
- §6.2.1: only `train.csv`, `test.csv`, and the competition-provided `smile_r3.csv` auxiliary
  file are read. No other dataset is attached.
- §6.2.2: every stage — including self-supervised pretraining — executes in this single run,
  top to bottom, no manual intervention. `SSL_MAX_HOURS` bounds the pretraining stage so the
  full run fits inside one Kaggle session; **verify your actual session time limit on Kaggle's
  site before setting it, it is not hard-coded here because it can change.**
- §6.2.4: no checkpoint, embedding, or artifact produced outside this run is ever loaded back
  in. The periodic `torch.save` calls during SSL pretraining are for your own inspection in the
  Kaggle output tab after a commit — nothing in this notebook reads them back as input.
- Random seeds are fixed throughout for §7.2 reproducibility.


In [ ]:
!pip install rdkit -q

In [ ]:
import os, sys, time, pickle, warnings, gc, re, glob
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem import (Descriptors, AllChem, MACCSkeys, rdMolDescriptors,
                        Lipinski, rdFingerprintGenerator)
RDLogger.logger().setLevel(RDLogger.ERROR)

import shap

In [ ]:
SEED         = 42
N_FOLDS      = 10
NN_SEEDS     = [42, 202, 777, 1337, 2024]   # averaging these is worth ~+0.015 on the NN alone
TARGET_TYPES = ['tg', 'egc', 'egb', 'eps', 'nc', 'ei', 'eea']
DFT_PROPS    = ['egc', 'egb', 'ei', 'eea', 'eps', 'nc']   # co-observed block ('tg' is disjoint)
MORGAN_BITS_R2, MORGAN_BITS_R3, AP_BITS, TT_BITS = 2048, 1024, 1024, 1024

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# NOTE: only the official competition input mount is searched -- no attached external datasets.
_KAGGLE_PATHS = ['/kaggle/input/competitions/ppp-round-3', '/kaggle/input/ppp-round-3',
                 '/kaggle/input/aisehack-2-0']
DATA_DIR = next((p for p in _KAGGLE_PATHS if os.path.exists(p)), os.getcwd())
WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

T_NOTEBOOK_START = time.time()   # tracked through the whole run for the time-budget checks below

class Logger:
    def __init__(self): self.t0 = time.time()
    def _p(self, lv, m):
        print(f'[{datetime.now().strftime("%H:%M:%S")}] [{lv:>6}] {m}', flush=True)
    def info(self, m): self._p('INFO', m)
    def metric(self, m): self._p('METRIC', m)
    def ok(self, m): self._p('OK', m)
    def warn(self, m): self._p('WARN', m)
    def header(self, m):
        el = (time.time() - T_NOTEBOOK_START) / 60
        self._p('INFO', '=' * 60)
        self._p('INFO', f'  {m}   [t+{el:.1f} min from notebook start]')
        self._p('INFO', '=' * 60)
    def sub(self, m): self._p('INFO', f'--- {m} ---')
log = Logger()

print(f'device={device}  data={DATA_DIR}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Hyperparameters

Capacity is scaled to sample count so the ~220-row properties don't get badly over-fit by settings sized for `tg` (4143 rows).

In [ ]:
LGBM_BASE = dict(objective='regression', metric='rmse', boosting_type='gbdt',
                 n_estimators=3000, learning_rate=0.015, max_depth=7, num_leaves=63,
                 min_child_samples=10, reg_alpha=0.1, reg_lambda=1.0,
                 subsample=0.8, colsample_bytree=0.6,
                 random_state=SEED, verbose=-1, n_jobs=-1)

XGB_BASE = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.015,
                max_depth=7, subsample=0.8, colsample_bytree=0.6,
                reg_alpha=0.1, reg_lambda=1.0, min_child_weight=10,
                random_state=SEED, verbosity=0)
if torch.cuda.is_available():
    XGB_BASE['device'] = 'cuda'

CB_BASE = dict(iterations=3000, learning_rate=0.03, depth=7, l2_leaf_reg=3.0,
               random_seed=SEED, verbose=0, od_type='Iter', od_wait=100)
if torch.cuda.is_available():
    CB_BASE['task_type'] = 'GPU'; CB_BASE['devices'] = '0'

SMALL = 600          # below this many rows, shrink the model

def lgbm_params(n):
    if n < SMALL:
        return dict(LGBM_BASE, num_leaves=7, max_depth=4, min_child_samples=5,
                    colsample_bytree=0.20, learning_rate=0.02, n_estimators=1500)
    return LGBM_BASE

def xgb_params(n):
    if n < SMALL:
        return dict(XGB_BASE, max_depth=3, min_child_weight=5,
                    colsample_bytree=0.20, learning_rate=0.02)
    return XGB_BASE

def cb_params(n):
    if n < SMALL:
        return dict(CB_BASE, depth=4, learning_rate=0.02)
    return CB_BASE

NN_CFG  = dict(hidden_dims=[1024, 512, 256, 128], head_dim=64, dropout=0.3,
               lr=1e-3, weight_decay=1e-4, epochs=200, batch_size=64, patience=25)
CNN_CFG = dict(embed_dim=64, n_filters=128, kernel_sizes=[3, 5, 7, 11], fc_dim=256,
               dropout=0.3, max_len=200, n_aug=5, lr=5e-4, weight_decay=1e-4,
               epochs=120, batch_size=64, patience=20)
print('hyperparameters set')

## 2. Load data

**Check the printed shapes against the Round 3 data page before trusting anything downstream** — the Round 3 brief states test.csv has 4,497 rows; the Round 2 run this pipeline is based on saw 4,940. If Round 3's file is genuinely a different size, that's expected; if it silently matches the old Round 2 file, something is wrong with `DATA_DIR`.

In [ ]:
log.header('LOADING DATA')
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

train_df = train_df.drop_duplicates(subset=['smiles', 'target_type', 'target'])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

log.info(f'train {train_df.shape}  test {test_df.shape}')
for tt in TARGET_TYPES:
    log.info(f'  {tt}: {(train_df.target_type==tt).sum()} train, '
             f'{(test_df.target_type==tt).sum()} test')

if len(test_df) != 4497:
    log.warn(f'test.csv has {len(test_df)} rows, Round 3 brief says 4,497 -- '
             f'confirm this is the intended file before proceeding')

## 3. Featurization

RDKit descriptors + Morgan(r=2,3) + AtomPair + Topological-Torsion + MACCS + polymer-specific terms + SMARTS functional-group counts.

In [ ]:
GROUP_SMARTS = {
    'aromatic_6': '[a]1[a][a][a][a][a]1', 'aromatic_5': '[a]1[a][a][a][a]1',
    'amide': '[NX3][CX3](=[OX1])', 'ester': '[CX3](=[OX1])[OX2]',
    'ether': '[OD2]([#6])[#6]', 'hydroxyl': '[OX2H]', 'carbonyl': '[CX3]=[OX1]',
    'carboxyl': '[CX3](=[OX1])[OX2H1]', 'sulfonyl': '[#16X4](=[OX1])(=[OX1])',
    'imide': '[CX3](=[OX1])[NX3][CX3](=[OX1])', 'urea': '[NX3][CX3](=[OX1])[NX3]',
    'cyano': '[CX2]#[NX1]', 'nitro': '[NX3+](=O)[O-]', 'fluorine': '[F]',
    'chlorine': '[Cl]', 'bromine': '[Br]', 'silicon': '[Si]', 'phosphorus': '[P]',
    'double_bond': '[CX3]=[CX3]', 'triple_bond': '[CX2]#[CX2]', 'epoxide': 'C1OC1',
    'azo': '[NX2]=[NX2]', 'thioether': '[#16X2]([#6])[#6]', 'amine_primary': '[NX3H2]',
    'amine_secondary': '[NX3H1]([#6])[#6]', 'amine_tertiary': '[NX3]([#6])([#6])[#6]',
    'phenol': '[OX2H][c]', 'vinyl': '[CX3]=[CX3H1]', 'methyl': '[CH3]',
    'trifluoromethyl': '[CX4](F)(F)F', 'anhydride': '[CX3](=[OX1])[OX2][CX3](=[OX1])',
}
GROUP_PATTERNS = {k: p for k, s in GROUP_SMARTS.items()
                  if (p := Chem.MolFromSmarts(s)) is not None}

def compute_custom(mol, smi):
    f = {}
    if mol is None: return f
    try:
        f['n_star'] = smi.count('*'); f['smi_len'] = len(smi)
        f['n_atoms'] = mol.GetNumAtoms(); f['n_heavy'] = mol.GetNumHeavyAtoms()
        f['n_bonds'] = mol.GetNumBonds()
        f['n_rings'] = mol.GetRingInfo().NumRings()
        f['n_arom_rings'] = rdMolDescriptors.CalcNumAromaticRings(mol)
        f['n_aliph_rings'] = rdMolDescriptors.CalcNumAliphaticRings(mol)
        f['arom_ratio'] = f['n_arom_rings'] / max(f['n_rings'], 1)
        f['ring_ratio'] = f['n_rings'] / max(f['n_atoms'], 1)
        f['n_rot'] = Lipinski.NumRotatableBonds(mol)
        f['rot_ratio'] = f['n_rot'] / max(f['n_bonds'], 1)
        f['n_het'] = Lipinski.NumHeteroatoms(mol)
        f['het_ratio'] = f['n_het'] / max(f['n_atoms'], 1)
        f['n_hbd'] = Lipinski.NumHDonors(mol); f['n_hba'] = Lipinski.NumHAcceptors(mol)
        f['fsp3'] = rdMolDescriptors.CalcFractionCSP3(mol)
        cj = sum(1 for b in mol.GetBonds() if b.GetIsConjugated())
        f['n_conj_bonds'] = cj; f['conj_ratio'] = cj / max(f['n_bonds'], 1)
        nums = [a.GetAtomicNum() for a in mol.GetAtoms()]
        for z, nm in [(6,'C'),(7,'N'),(8,'O'),(9,'F'),(16,'S'),(17,'Cl'),(35,'Br'),(14,'Si'),(15,'P')]:
            f[f'n_{nm}'] = nums.count(z); f[f'fr_{nm}'] = nums.count(z)/max(len(nums),1)
        stars = [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == '*']
        if len(stars) == 2:
            try:
                path = Chem.rdmolops.GetShortestPath(mol, stars[0], stars[1])
                f['backbone_len'] = len(path) - 2
                ba = sum(1 for i in path[1:-1] if mol.GetAtomWithIdx(i).GetIsAromatic())
                f['backbone_arom_ratio'] = ba / max(f['backbone_len'], 1)
            except Exception:
                f['backbone_len'] = 0; f['backbone_arom_ratio'] = 0.0
        try:
            AllChem.ComputeGasteigerCharges(mol)
            ch = [a.GetDoubleProp('_GasteigerCharge') for a in mol.GetAtoms()]
            ch = [c for c in ch if np.isfinite(c)]
            if ch:
                f['ch_mean']=np.mean(ch); f['ch_std']=np.std(ch)
                f['ch_min']=np.min(ch);  f['ch_max']=np.max(ch)
                f['ch_range']=f['ch_max']-f['ch_min']
        except Exception: pass
        for nm, fn in [('balaban_j', Descriptors.BalabanJ), ('bertz_ct', Descriptors.BertzCT)]:
            try: f[nm] = fn(mol)
            except Exception: pass
    except Exception: pass
    return {k: (0.0 if v is None or (isinstance(v,float) and not np.isfinite(v)) else v)
            for k, v in f.items()}

def featurize_batch(smiles_list):
    n = len(smiles_list); t0 = time.time(); every = max(1, n//10)
    rd_l, m2_l, m3_l, ap_l, tt_l, mc_l, cu_l, gr_l = [], [], [], [], [], [], [], []
    apg = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=AP_BITS)
    ttg = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=TT_BITS)
    log.info(f'featurizing {n} molecules...')
    for i, smi in enumerate(smiles_list):
        if (i+1) % every == 0:
            el = time.time()-t0
            log.info(f'  {i+1}/{n} ({100*(i+1)/n:.0f}%) ETA {(n-i-1)/max((i+1)/el,.01):.0f}s')
        mol = Chem.MolFromSmiles(smi)
        try:
            d = Descriptors.CalcMolDescriptors(mol) if mol is not None else {}
            rd_l.append({k: (0.0 if v is None or (isinstance(v,float) and not np.isfinite(v))
                             else float(v)) for k, v in d.items()})
        except Exception:
            rd_l.append({})
        if mol is not None:
            m2_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=MORGAN_BITS_R2), dtype=np.float32))
            m3_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol,3,nBits=MORGAN_BITS_R3), dtype=np.float32))
            ap_l.append(apg.GetFingerprintAsNumPy(mol).astype(np.float32))
            tt_l.append(ttg.GetFingerprintAsNumPy(mol).astype(np.float32))
            mc_l.append(np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32))
        else:
            m2_l.append(np.zeros(MORGAN_BITS_R2, np.float32)); m3_l.append(np.zeros(MORGAN_BITS_R3, np.float32))
            ap_l.append(np.zeros(AP_BITS, np.float32));        tt_l.append(np.zeros(TT_BITS, np.float32))
            mc_l.append(np.zeros(167, np.float32))
        cu_l.append(compute_custom(mol, smi))
        gr_l.append({f'grp_{k}': len(mol.GetSubstructMatches(p)) if mol is not None else 0
                     for k, p in GROUP_PATTERNS.items()})
    log.info(f'done in {time.time()-t0:.0f}s')
    df_rd = pd.DataFrame(rd_l); df_rd.columns = [f'rd_{c}' for c in df_rd.columns]
    df_cu = pd.DataFrame(cu_l); df_cu.columns = [f'po_{c}' for c in df_cu.columns]
    parts = [df_rd,
             pd.DataFrame(np.array(m2_l), columns=[f'mfp2_{i}' for i in range(MORGAN_BITS_R2)]),
             pd.DataFrame(np.array(m3_l), columns=[f'mfp3_{i}' for i in range(MORGAN_BITS_R3)]),
             pd.DataFrame(np.array(ap_l), columns=[f'ap_{i}' for i in range(AP_BITS)]),
             pd.DataFrame(np.array(tt_l), columns=[f'tt_{i}' for i in range(TT_BITS)]),
             pd.DataFrame(np.array(mc_l), columns=[f'mac_{i}' for i in range(167)]),
             df_cu, pd.DataFrame(gr_l)]
    return pd.concat(parts, axis=1)

def clean_features(df):
    df = df.copy()
    fm = float(np.finfo(np.float32).max)
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].clip(lower=-fm, upper=fm)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0.0, inplace=True)
    obj = df.select_dtypes(include=['object']).columns.tolist()
    if obj: df.drop(columns=obj, inplace=True)
    df.columns = [re.sub(r'[\[\]<,]', '_', c) for c in df.columns]
    return df

log.header('FEATURE ENGINEERING')
CACHE = os.path.join(WORK_DIR, 'features_final.pkl')
if os.path.exists(CACHE):
    train_features, test_features = pickle.load(open(CACHE, 'rb'))
    log.ok(f'loaded cached features {train_features.shape}')
else:
    all_smi = pd.concat([train_df[['smiles']], test_df[['smiles']]]).drop_duplicates('smiles')
    feats = clean_features(featurize_batch(all_smi['smiles'].tolist()))
    feats.index = all_smi['smiles'].values
    train_features = feats.loc[train_df.smiles.values].reset_index(drop=True)
    test_features  = feats.loc[test_df.smiles.values].reset_index(drop=True)
    const = train_features.columns[train_features.nunique() <= 1].tolist()
    if const:
        train_features.drop(columns=const, inplace=True)
        test_features.drop(columns=[c for c in const if c in test_features.columns], inplace=True)
        log.info(f'dropped {len(const)} constant columns')
    del feats; gc.collect()
    pickle.dump((train_features, test_features), open(CACHE, 'wb'), protocol=4)
log.info(f'train_features {train_features.shape}  test_features {test_features.shape}')

## 4. TRUE co-observed partner features + physics

Built from `train.csv` only. The runtime assertion at the end is the archive guard required by §6.2.1.

In [ ]:
log.header('TRUE PARTNER FEATURES')
USES  = {}         # engineered column -> set of properties whose LABEL it uses
FILLS = {}         # column -> fill value computed on TRAIN (applied to train and test alike)

def _canon(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m) if m is not None else s

_cmap = {s: _canon(s) for s in set(train_df.smiles) | set(test_df.smiles)}
_tc = train_df.smiles.map(_cmap)
_ec = test_df.smiles.map(_cmap)
log.info(f'canonical molecules: {len(set(_cmap.values()))} of {len(_cmap)} raw SMILES')

_tmp = train_df.assign(_c=_tc)
_truth = {q: _tmp[_tmp.target_type == q].groupby('_c').target.mean() for q in DFT_PROPS}
log.info('partner table: ' + ', '.join(f'{q}={len(_truth[q])}' for q in DFT_PROPS))

raw_tr, raw_te = {}, {}
for q in DFT_PROPS:
    raw_tr[q] = _tc.map(_truth[q]).values.astype(np.float64)
    raw_te[q] = _ec.map(_truth[q]).values.astype(np.float64)

def _add(name, a, b, srcs):
    fill = float(np.nanmean(np.where(np.isfinite(a), a, np.nan)))
    FILLS[name] = fill
    train_features[name] = np.where(np.isfinite(a), a, fill)
    test_features[name]  = np.where(np.isfinite(b), b, fill)
    train_features[f'{name}_ok'] = np.isfinite(a).astype(np.float32)
    test_features[f'{name}_ok']  = np.isfinite(b).astype(np.float32)
    FILLS[f'{name}_ok'] = 0.0
    USES[name] = set(srcs); USES[f'{name}_ok'] = set(srcs)

for q in DFT_PROPS:
    _add(f'true_{q}', raw_tr[q], raw_te[q], [q])

_add('ph_ei',  raw_tr['egc']+raw_tr['eea'], raw_te['egc']+raw_te['eea'], ['egc','eea'])
_add('ph_eea', raw_tr['ei']-raw_tr['egc'],  raw_te['ei']-raw_te['egc'],  ['ei','egc'])
_add('ph_egb', raw_tr['egc'],               raw_te['egc'],               ['egc'])
_add('ph_eps', raw_tr['nc']**2,             raw_te['nc']**2,             ['nc'])
_add('ph_nc',  np.sqrt(np.clip(raw_tr['eps'],0,None)),
               np.sqrt(np.clip(raw_te['eps'],0,None)),                   ['eps'])
_add('ph_gap', raw_tr['egb']-raw_tr['egc'], raw_te['egb']-raw_te['egc'], ['egb','egc'])

def drop_leaky(feat_df, target_type):
    bad = [c for c, s in USES.items() if target_type in s and c in feat_df.columns]
    return feat_df.drop(columns=bad)

def mask_rows_for_multitask(feat_df, target_types):
    out = feat_df.copy()
    tt = np.asarray(target_types)
    for c, s in USES.items():
        if c in out.columns:
            m = np.isin(tt, list(s))
            if m.any():
                out.loc[m, c] = FILLS[c]
    return out

for q in DFT_PROPS:
    assert len(_truth[q]) == _tmp[_tmp.target_type == q]._c.nunique(), \
        f'{q}: partner table exceeds train.csv -- external labels merged in'
log.ok('COMPLIANCE: partner table built from train.csv only')
log.ok(f'added {len(USES)} engineered columns -> {train_features.shape[1]} features total')

## 5. Leakage assertions

Proves the leak is real first, then proves each guard removes it. Aborts on any failure.

In [ ]:
log.header('LEAKAGE SELF-TEST')
fail = []

for p in DFT_PROPS:
    m = (train_df.target_type == p).values
    if not np.allclose(train_features.loc[m, f'true_{p}'], train_df.loc[m, 'target'], atol=1e-6):
        fail.append(f'{p}: true_{p} does NOT reproduce target -- feature build is wrong')
log.info('(a) leak reproduced for all DFT properties (as expected)')

for p in TARGET_TYPES:
    kept = drop_leaky(train_features, p)
    bad = [c for c in kept.columns if p in USES.get(c, set())]
    if bad: fail.append(f'{p}: drop_leaky left {bad}')
log.info('(b) drop_leaky leaves no dependent column')

_mm = mask_rows_for_multitask(train_features, train_df.target_type.values)
for p in DFT_PROPS:
    m = (train_df.target_type == p).values
    if np.allclose(_mm.loc[m, f'true_{p}'], train_df.loc[m, 'target'], atol=1e-6):
        fail.append(f'{p}: NN row-mask did not neutralise true_{p}')
log.info('(c) NN row-mask neutralises own-target columns')

log.info('(d) partner availability, train vs test (must be close or CV will not transfer):')
for p in DFT_PROPS:
    mtr = (train_df.target_type == p).values
    mte = (test_df.target_type == p).values
    cols = [f'true_{q}_ok' for q in DFT_PROPS if q != p]
    a = train_features.loc[mtr, cols].sum(1).mean()
    b = test_features.loc[mte,  cols].sum(1).mean()
    flag = '  <-- CHECK' if abs(a - b) > 0.5 else ''
    log.info(f'    {p:4s} mean partner count  train {a:.2f}  test {b:.2f}{flag}')

assert not fail, 'LEAKAGE CHECK FAILED:\n' + '\n'.join(fail)
log.ok('all leakage checks passed')

## 6. v2 config — score-improvement techniques

LB (0.883) came in well below OOF (~0.903) on v1. `USE_SSL = False` below turns off the
self-supervised branch by default — that's the prime suspect for the gap and should be
re-enabled only after you've confirmed on LB that leaving it out helps. Everything else here is
additive to the Round 2 baseline.

In [ ]:
USE_SSL             = False   # diagnostic: v1 LB dropped after adding this; off until proven to help
PSEUDO_LABEL         = False  # risky, off by default -- turn on only after everything else is validated
SMALL_N_FOLDS        = 15     # more folds -> less noisy OOF on the ~220-row properties
FEATURE_SELECT_TOPK  = 500    # cap features per fold for small-n properties (5400 features / 220 rows is absurd)
SHRINK_DEFAULT       = 0.75
SHRINK_OVERRIDE      = {'ei': 0.90, 'eea': 0.90}   # physics R2 is far above the stack here (.963/.971 vs ~.88)
CNN_SEEDS            = [42, 202, 777, 1337, 2024]  # was single-seed in v1; mirror the NN's averaging
MPNN_SEEDS           = [42, 202, 777]
PSEUDO_CONF_STD      = 0.15   # z-scored units; only pseudo-label test rows where base models agree this tightly
PSEUDO_MIN_N         = 25
log.info(f'USE_SSL={USE_SSL}  PSEUDO_LABEL={PSEUDO_LABEL}  SMALL_N_FOLDS={SMALL_N_FOLDS}  '
         f'FEATURE_SELECT_TOPK={FEATURE_SELECT_TOPK}')

## 7. LightGBM / XGBoost / CatBoost — with per-fold feature selection + adaptive folds

Two changes from v1: (a) properties with n < `SMALL` get `SMALL_N_FOLDS` instead of 10 —
more, smaller validation splits reduce OOF noise when you only have ~220 rows; (b) those same
properties get a per-fold univariate feature filter down to `FEATURE_SELECT_TOPK` columns.
Selection is fit **inside each fold on the training split only** — fitting it on the full
property's data first and then cross-validating would leak validation-row information into
the selection and overstate OOF.

In [ ]:
log.header('GRADIENT BOOSTING (v2: feature-selected, adaptive folds)')

def select_features(X, y, topk):
    if X.shape[1] <= topk:
        return X.columns.tolist()
    yv = y.values if hasattr(y, 'values') else y
    std = X.std()
    corr = X.apply(lambda col: abs(np.corrcoef(col, yv)[0, 1]) if col.std() > 1e-9 else 0.0)
    corr = corr.fillna(0.0)
    return corr.sort_values(ascending=False).head(topk).index.tolist()

def cv_tree(kind, X, y, tt, select_topk=None, n_folds=None):
    n = len(y)
    nf = n_folds or N_FOLDS
    kf = KFold(nf, shuffle=True, random_state=SEED)
    oof = np.zeros(n); models = []; fold_cols = []
    for f, (a, b) in enumerate(kf.split(X)):
        Xa, Xb = X.iloc[a], X.iloc[b]
        ya, yb = y.iloc[a], y.iloc[b]
        cols = None
        if select_topk is not None and X.shape[1] > select_topk:
            cols = select_features(Xa, ya, select_topk)
            Xa, Xb = Xa[cols], Xb[cols]
        if kind == 'lgbm':
            m = lgb.LGBMRegressor(**lgbm_params(n))
            m.fit(Xa, ya, eval_set=[(Xb, yb)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        elif kind == 'xgb':
            m = xgb.XGBRegressor(early_stopping_rounds=100, **xgb_params(n))
            m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        else:
            m = cb.CatBoostRegressor(**cb_params(n))
            m.fit(Xa.values, ya.values, eval_set=(Xb.values, yb.values), verbose=0)
        oof[b] = m.predict(Xb.values if kind == 'cb' else Xb)
        models.append(m); fold_cols.append(cols)
    r2 = r2_score(y, oof)
    log.metric(f'  [{tt}] {kind} OOF R2={r2:.4f}  (n={n}, folds={nf}, '
               f'{"selected-"+str(select_topk) if select_topk else "all"} features)')
    return models, oof, r2, fold_cols

tree_models, tree_oof, tree_r2, tree_cols = {}, {}, {}, {}
for kind in ['lgbm', 'xgb', 'cb']:
    log.sub(kind)
    tree_models[kind] = {}; tree_r2[kind] = {}; tree_cols[kind] = {}
    oof_all = np.zeros(len(train_df))
    for tt in TARGET_TYPES:
        mask = (train_df.target_type == tt).values
        X = drop_leaky(train_features[mask].reset_index(drop=True), tt)
        y = train_df.loc[mask, 'target'].reset_index(drop=True)
        n = len(y)
        topk = FEATURE_SELECT_TOPK if n < SMALL else None
        nf = SMALL_N_FOLDS if n < SMALL else N_FOLDS
        mods, oof, r2, cols = cv_tree(kind, X, y, tt, select_topk=topk, n_folds=nf)
        tree_models[kind][tt] = mods; tree_r2[kind][tt] = r2; tree_cols[kind][tt] = cols
        oof_all[mask] = oof
    tree_oof[kind] = oof_all
    log.metric(f'>>> {kind} mean OOF R2 = {np.mean(list(tree_r2[kind].values())):.4f}')

## 8. Multi-task neural network (5-seed averaged) — unchanged from v1

In [ ]:
log.header('MULTI-TASK NN')
task_map = {t: i for i, t in enumerate(TARGET_TYPES)}

class MultiTaskNet(nn.Module):
    def __init__(s, d_in, hidden=(1024,512,256,128), head=64, n_tasks=7, dropout=0.3):
        super().__init__()
        L, prev = [], d_in
        for i, h in enumerate(hidden):
            L += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.SiLU(),
                  nn.Dropout(max(dropout*(1-i*0.1), 0.05))]
            prev = h
        s.trunk = nn.Sequential(*L)
        s.heads = nn.ModuleList([nn.Sequential(nn.Linear(prev,head), nn.SiLU(),
                                               nn.Dropout(dropout*0.3), nn.Linear(head,1))
                                 for _ in range(n_tasks)])
    def forward(s, x, t):
        z = s.trunk(x)
        out = torch.zeros(x.size(0), device=x.device)
        for i, h in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = h(z[m]).squeeze(-1)
        return out

class DS(Dataset):
    def __init__(s, X, y, t):
        s.X = torch.FloatTensor(X); s.y = torch.FloatTensor(y); s.t = torch.LongTensor(t)
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return s.X[i], s.y[i], s.t[i]

X_nn_train = mask_rows_for_multitask(train_features, train_df.target_type.values).values
X_nn_test  = mask_rows_for_multitask(test_features,  test_df.target_type.values).values
X_nn_train = np.nan_to_num(np.clip(X_nn_train, -3.4e38, 3.4e38)).astype(np.float32)
X_nn_test  = np.nan_to_num(np.clip(X_nn_test,  -3.4e38, 3.4e38)).astype(np.float32)
log.ok('NN inputs row-masked')

y_all  = train_df.target.values.astype(np.float32)
t_all  = train_df.target_type.map(task_map).values.astype(np.int64)
t_test = test_df.target_type.map(task_map).values.astype(np.int64)

nn_oof = np.zeros(len(y_all)); nn_fold_models = []

for _si, _sd in enumerate(NN_SEEDS):
  _oof_seed = np.zeros(len(y_all))
  for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(X_nn_train)):
    t0 = time.time(); torch.manual_seed(_sd + fold)
    sc = StandardScaler()
    Xa = np.nan_to_num(sc.fit_transform(X_nn_train[tr_i])).astype(np.float32)
    Xb = np.nan_to_num(sc.transform(X_nn_train[va_i])).astype(np.float32)
    tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
    for tt, i in task_map.items():
        ma, mb = (t_all[tr_i] == i), (t_all[va_i] == i)
        s = StandardScaler()
        if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
        if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
        tsc[i] = s
    model = MultiTaskNet(Xa.shape[1], NN_CFG['hidden_dims'], NN_CFG['head_dim'],
                         dropout=NN_CFG['dropout']).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=NN_CFG['lr'], weight_decay=NN_CFG['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NN_CFG['epochs'], eta_min=1e-6)
    dl  = DataLoader(DS(Xa, ya, t_all[tr_i]), batch_size=NN_CFG['batch_size'],
                     shuffle=True, drop_last=True)
    vdl = DataLoader(DS(Xb, yb, t_all[va_i]), batch_size=NN_CFG['batch_size']*4)
    best, best_state, pat = 1e18, None, 0
    for ep in range(NN_CFG['epochs']):
        model.train()
        for xb, yy, tb in dl:
            xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
            opt.zero_grad()
            loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
            if torch.isnan(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for xb, yy, tb in vdl:
                xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
                vl += F.mse_loss(model(xb, tb), yy).item()*len(xb); nv += len(xb)
        vl /= max(nv,1); sch.step()
        if vl < best - 1e-6:
            best, best_state, pat = vl, {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= NN_CFG['patience']: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pn = model(torch.FloatTensor(Xb).to(device),
                   torch.LongTensor(t_all[va_i]).to(device)).cpu().numpy()
    pred = np.zeros_like(pn)
    for tt, i in task_map.items():
        m = (t_all[va_i] == i)
        if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
    _oof_seed[va_i] = pred
    nn_fold_models.append((sc, tsc, model))
    log.metric(f'  NN seed {_si+1}/{len(NN_SEEDS)} fold {fold+1}/{N_FOLDS}: {time.time()-t0:.0f}s')

  _r = np.mean([r2_score(y_all[t_all==i], _oof_seed[t_all==i]) for i in task_map.values()])
  log.metric(f'  >> seed {_sd} mean OOF R2 = {_r:.4f}')
  nn_oof += _oof_seed / len(NN_SEEDS)

assert len(nn_fold_models) == len(NN_SEEDS) * N_FOLDS
nn_r2 = {tt: r2_score(y_all[t_all==i], nn_oof[t_all==i]) for tt, i in task_map.items()}
for tt in TARGET_TYPES: log.metric(f'  NN [{tt}] R2={nn_r2[tt]:.4f}')
log.metric(f'>>> NN mean OOF R2 = {np.mean(list(nn_r2.values())):.4f}')

## 9. SMILES 1D-CNN — now 5-seed averaged (v1 was single-seed)

Same architecture as v1; the only change is wrapping the existing fold loop in a seed loop and
averaging, mirroring what already works for the NN (~+0.015 R2 there in Round 2). The CNN was
the noisiest single model in v1, so it's a reasonable bet this transfers.

In [ ]:
log.header('SMILES CNN (5-seed averaged)')
SMILES_CHARS = list("CNOFPSIBrclnos=#()-+[]@12345678/\\.%*{}~<>^ ")
C2I = {c: i+1 for i, c in enumerate(SMILES_CHARS)}
VOCAB = len(SMILES_CHARS) + 1

def tok(s, L=CNN_CFG['max_len']):
    t = [C2I.get(c, 0) for c in s[:L]]
    return t + [0]*(L-len(t))

def aug(s, n):
    m = Chem.MolFromSmiles(s)
    if m is None: return [s]*n
    out = set()
    for _ in range(n*5):
        try: out.add(Chem.MolToSmiles(m, doRandom=True))
        except Exception: pass
        if len(out) >= n: break
    r = list(out)[:n]
    return r + [s]*(n-len(r))

class CNN(nn.Module):
    def __init__(s, p=0.3):
        super().__init__()
        s.emb = nn.Embedding(VOCAB, CNN_CFG['embed_dim'], padding_idx=0)
        s.convs = nn.ModuleList([nn.Sequential(
            nn.Conv1d(CNN_CFG['embed_dim'], CNN_CFG['n_filters'], k, padding=k//2),
            nn.BatchNorm1d(CNN_CFG['n_filters']), nn.SiLU()) for k in CNN_CFG['kernel_sizes']])
        pd_ = CNN_CFG['n_filters']*len(CNN_CFG['kernel_sizes'])*2
        s.fc = nn.Sequential(nn.Linear(pd_, CNN_CFG['fc_dim']), nn.BatchNorm1d(CNN_CFG['fc_dim']),
                             nn.SiLU(), nn.Dropout(p))
        s.heads = nn.ModuleList([nn.Sequential(nn.Linear(CNN_CFG['fc_dim'],64), nn.SiLU(),
                                               nn.Dropout(p*0.3), nn.Linear(64,1)) for _ in range(7)])
    def forward(s, x, t):
        e = s.emb(x).transpose(1,2); o = []
        for c in s.convs:
            z = c(e); o.append(z.mean(2)); o.append(z.max(2).values)
        h = s.fc(torch.cat(o, 1))
        out = torch.zeros(x.size(0), device=x.device)
        for i, hd in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = hd(h[m]).squeeze(-1)
        return out

class SDS(Dataset):
    def __init__(s, smi, y, t, augment=False, n=1):
        if augment and n > 1:
            X, Y, T = [], [], []
            for a, b, c in zip(smi, y, t):
                for v in aug(a, n): X.append(tok(v)); Y.append(b); T.append(c)
        else:
            X, Y, T = [tok(v) for v in smi], list(y), list(t)
        s.X = torch.LongTensor(X); s.y = torch.FloatTensor(Y); s.t = torch.LongTensor(T)
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return s.X[i], s.y[i], s.t[i]

smi_all = train_df.smiles.values
cnn_oof = np.zeros(len(y_all)); cnn_fold_models = []

for _si, _sd in enumerate(CNN_SEEDS):
  _oof_seed = np.zeros(len(y_all))
  for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(smi_all)):
    t0 = time.time(); torch.manual_seed(_sd + fold)
    tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
    for tt, i in task_map.items():
        ma, mb = (t_all[tr_i]==i), (t_all[va_i]==i)
        s = StandardScaler()
        if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
        if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
        tsc[i] = s
    model = CNN(CNN_CFG['dropout']).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CNN_CFG['lr'], weight_decay=CNN_CFG['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_CFG['epochs'], eta_min=1e-6)
    dl  = DataLoader(SDS(smi_all[tr_i], ya, t_all[tr_i], True, CNN_CFG['n_aug']),
                     batch_size=CNN_CFG['batch_size'], shuffle=True, drop_last=True)
    vdl = DataLoader(SDS(smi_all[va_i], yb, t_all[va_i]), batch_size=CNN_CFG['batch_size']*4)
    best, best_state, pat = 1e18, None, 0
    for ep in range(CNN_CFG['epochs']):
        model.train()
        for xb, yy, tb in dl:
            xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
            opt.zero_grad()
            loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
            if torch.isnan(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for xb, yy, tb in vdl:
                xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
                vl += F.mse_loss(model(xb, tb), yy).item()*len(xb); nv += len(xb)
        vl /= max(nv,1); sch.step()
        if vl < best - 1e-6:
            best, best_state, pat = vl, {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= CNN_CFG['patience']: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    vds = SDS(smi_all[va_i], yb, t_all[va_i]); pl = []
    with torch.no_grad():
        for xb, _, tb in DataLoader(vds, batch_size=512):
            pl.append(model(xb.to(device), tb.to(device)).cpu().numpy())
    pn = np.concatenate(pl); pred = np.zeros_like(pn)
    for tt, i in task_map.items():
        m = (t_all[va_i]==i)
        if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
    _oof_seed[va_i] = pred; cnn_fold_models.append((tsc, model))
    log.metric(f'  CNN seed {_si+1}/{len(CNN_SEEDS)} fold {fold+1}/{N_FOLDS}: {time.time()-t0:.0f}s')

  _r = np.mean([r2_score(y_all[t_all==i], _oof_seed[t_all==i]) for i in task_map.values()])
  log.metric(f'  >> seed {_sd} mean OOF R2 = {_r:.4f}')
  cnn_oof += _oof_seed / len(CNN_SEEDS)

cnn_r2 = {tt: r2_score(y_all[t_all==i], cnn_oof[t_all==i]) for tt, i in task_map.items()}
for tt in TARGET_TYPES: log.metric(f'  CNN [{tt}] R2={cnn_r2[tt]:.4f}')
log.metric(f'>>> CNN mean OOF R2 = {np.mean(list(cnn_r2.values())):.4f}  (compare to v1 single-seed)')

## 10. MPNN — message-passing graph network (reintroduced, pure PyTorch)

The base model type consistently flagged as the strongest expected contributor and the one
missing from v1. Implemented with **dense padded adjacency matrices** rather than
PyTorch Geometric (kept to plain `torch.nn`, no extra graph library): each molecule becomes a
fixed-size `[MAX_ATOMS, MAX_ATOMS]` adjacency/edge-feature tensor, message passing is a masked
matrix multiply, and a GRU cell updates atom states — a standard MPNN, just batched by padding
instead of by a graph-library's sparse batching.

Multi-task, same z-scoring pattern as the NN/CNN. `MPNN_SEEDS` defaults to 3 (vs the NN's 5) to
keep this within the runtime budget — raise it if you have hours to spare.

In [ ]:
log.header('MPNN FEATURIZATION')

MAX_ATOMS = 60
ATOM_VOCAB = {z: i for i, z in enumerate([6,7,8,9,15,16,17,35,53,14,0], start=1)}  # C N O F P S Cl Br I Si *; 0=other
ATOM_VOCAB_SIZE = len(ATOM_VOCAB) + 2   # +1 for "other", +1 for pad(0)
ATOM_CONT_DIM = 6    # degree, formal_charge, aromatic, in_ring, num_H, hybridization_ord
EDGE_DIM = 6          # single, double, triple, aromatic, conjugated, in_ring

_HYB_ORD = {Chem.rdchem.HybridizationType.SP: 1, Chem.rdchem.HybridizationType.SP2: 2,
            Chem.rdchem.HybridizationType.SP3: 3, Chem.rdchem.HybridizationType.SP3D: 4,
            Chem.rdchem.HybridizationType.SP3D2: 5}

def mol_to_graph(smi, max_atoms=MAX_ATOMS):
    mol = Chem.MolFromSmiles(smi)
    atom_idx = np.zeros(max_atoms, dtype=np.int64)
    atom_cont = np.zeros((max_atoms, ATOM_CONT_DIM), dtype=np.float32)
    edge_feat = np.zeros((max_atoms, max_atoms, EDGE_DIM), dtype=np.float32)
    atom_mask = np.zeros(max_atoms, dtype=np.float32)
    if mol is None:
        return atom_idx, atom_cont, edge_feat, atom_mask
    n = min(mol.GetNumAtoms(), max_atoms)
    for i in range(n):
        a = mol.GetAtomWithIdx(i)
        atom_idx[i] = ATOM_VOCAB.get(a.GetAtomicNum(), len(ATOM_VOCAB) + 1)
        atom_cont[i] = [a.GetDegree(), a.GetFormalCharge(), float(a.GetIsAromatic()),
                        float(a.IsInRing()), a.GetTotalNumHs(), _HYB_ORD.get(a.GetHybridization(), 0)]
        atom_mask[i] = 1.0
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if i >= max_atoms or j >= max_atoms: continue
        bt = b.GetBondType()
        v = [float(bt == Chem.BondType.SINGLE), float(bt == Chem.BondType.DOUBLE),
             float(bt == Chem.BondType.TRIPLE), float(bt == Chem.BondType.AROMATIC),
             float(b.GetIsConjugated()), float(b.IsInRing())]
        edge_feat[i, j] = v; edge_feat[j, i] = v
    return atom_idx, atom_cont, edge_feat, atom_mask

def build_graph_batch(smiles_list):
    ai, ac, ef, am = zip(*[mol_to_graph(s) for s in smiles_list])
    return (torch.LongTensor(np.stack(ai)), torch.FloatTensor(np.stack(ac)),
            torch.FloatTensor(np.stack(ef)), torch.FloatTensor(np.stack(am)))

log.ok(f'graph featurization ready: MAX_ATOMS={MAX_ATOMS}, atom_vocab={ATOM_VOCAB_SIZE}, edge_dim={EDGE_DIM}')

In [ ]:
class MPLayer(nn.Module):
    def __init__(s, hidden, edge_dim):
        super().__init__()
        s.msg = nn.Sequential(nn.Linear(hidden + edge_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        s.gru = nn.GRUCell(hidden, hidden)

    def forward(s, h, edge_feat, adj_mask):
        B, N, H = h.shape
        h_u = h.unsqueeze(1).expand(B, N, N, H)                 # h_u[b,v,u,:] = h[b,u,:]
        msg_in = torch.cat([h_u, edge_feat], dim=-1)             # [B,N,N,H+E]
        m = s.msg(msg_in) * adj_mask.unsqueeze(-1)                # zero out non-edges
        agg = m.sum(dim=2)                                        # [B,N,H] -- sum over neighbors u
        h_flat = h.reshape(B * N, H)
        agg_flat = agg.reshape(B * N, H)
        h_new = s.gru(agg_flat, h_flat).reshape(B, N, H)
        return h_new

class MPNN(nn.Module):
    def __init__(s, hidden=128, n_layers=4, n_tasks=7, dropout=0.2):
        super().__init__()
        s.atom_emb = nn.Embedding(ATOM_VOCAB_SIZE, hidden, padding_idx=0)
        s.cont_proj = nn.Linear(ATOM_CONT_DIM, hidden)
        s.layers = nn.ModuleList([MPLayer(hidden, EDGE_DIM) for _ in range(n_layers)])
        s.readout = nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(), nn.Dropout(dropout))
        s.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_tasks)])

    def forward(s, atom_idx, atom_cont, edge_feat, atom_mask, adj_mask, t):
        h = s.atom_emb(atom_idx) + s.cont_proj(atom_cont)
        h = h * atom_mask.unsqueeze(-1)
        for layer in s.layers:
            h = layer(h, edge_feat, adj_mask)
            h = h * atom_mask.unsqueeze(-1)
        pooled = (h * atom_mask.unsqueeze(-1)).sum(1) / atom_mask.sum(1, keepdim=True).clamp(min=1)
        z = s.readout(pooled)
        out = torch.zeros(atom_idx.size(0), device=atom_idx.device)
        for i, hd in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = hd(z[m]).squeeze(-1)
        return out

class GraphDS(Dataset):
    def __init__(s, smi, y, t):
        ai, ac, ef, am = build_graph_batch(list(smi))
        s.ai, s.ac, s.ef, s.am = ai, ac, ef, am
        s.adj = (ef.abs().sum(-1) > 0).float()
        s.y = torch.FloatTensor(y); s.t = torch.LongTensor(t)
    def __len__(s): return len(s.y)
    def __getitem__(s, i): return s.ai[i], s.ac[i], s.ef[i], s.am[i], s.adj[i], s.y[i], s.t[i]

log.ok('MPNN model + dataset defined')

In [ ]:
log.header('MPNN TRAINING')
MPNN_CFG = dict(hidden=128, n_layers=4, dropout=0.2, lr=8e-4, weight_decay=1e-4,
                epochs=150, batch_size=48, patience=20)

mpnn_oof = np.zeros(len(y_all)); mpnn_fold_models = []

for _si, _sd in enumerate(MPNN_SEEDS):
  _oof_seed = np.zeros(len(y_all))
  for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(smi_all)):
    t0 = time.time(); torch.manual_seed(_sd + fold)
    tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
    for tt, i in task_map.items():
        ma, mb = (t_all[tr_i]==i), (t_all[va_i]==i)
        s = StandardScaler()
        if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
        if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
        tsc[i] = s

    tr_ds = GraphDS(smi_all[tr_i], ya, t_all[tr_i])
    va_ds = GraphDS(smi_all[va_i], yb, t_all[va_i])
    dl  = DataLoader(tr_ds, batch_size=MPNN_CFG['batch_size'], shuffle=True, drop_last=True)
    vdl = DataLoader(va_ds, batch_size=MPNN_CFG['batch_size']*2)

    model = MPNN(MPNN_CFG['hidden'], MPNN_CFG['n_layers'], dropout=MPNN_CFG['dropout']).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=MPNN_CFG['lr'], weight_decay=MPNN_CFG['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MPNN_CFG['epochs'], eta_min=1e-6)
    best, best_state, pat = 1e18, None, 0

    for ep in range(MPNN_CFG['epochs']):
        model.train()
        for ai, ac, ef, am, adj, yy, tb in dl:
            ai,ac,ef,am,adj,yy,tb = [x.to(device) for x in (ai,ac,ef,am,adj,yy,tb)]
            opt.zero_grad()
            loss = F.huber_loss(model(ai,ac,ef,am,adj,tb), yy, delta=1.0)
            if torch.isnan(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for ai, ac, ef, am, adj, yy, tb in vdl:
                ai,ac,ef,am,adj,yy,tb = [x.to(device) for x in (ai,ac,ef,am,adj,yy,tb)]
                vl += F.mse_loss(model(ai,ac,ef,am,adj,tb), yy).item()*len(yy); nv += len(yy)
        vl /= max(nv,1); sch.step()
        if vl < best - 1e-6:
            best, best_state, pat = vl, {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= MPNN_CFG['patience']: break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    pl = []
    with torch.no_grad():
        for ai, ac, ef, am, adj, yy, tb in DataLoader(va_ds, batch_size=256):
            ai,ac,ef,am,adj,tb = [x.to(device) for x in (ai,ac,ef,am,adj,tb)]
            pl.append(model(ai,ac,ef,am,adj,tb).cpu().numpy())
    pn = np.concatenate(pl); pred = np.zeros_like(pn)
    for tt, i in task_map.items():
        m = (t_all[va_i]==i)
        if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
    _oof_seed[va_i] = pred; mpnn_fold_models.append((tsc, model))
    log.metric(f'  MPNN seed {_si+1}/{len(MPNN_SEEDS)} fold {fold+1}/{N_FOLDS}: {time.time()-t0:.0f}s, val_mse={best:.4f}')

  _r = np.mean([r2_score(y_all[t_all==i], _oof_seed[t_all==i]) for i in task_map.values()])
  log.metric(f'  >> seed {_sd} mean OOF R2 = {_r:.4f}')
  mpnn_oof += _oof_seed / len(MPNN_SEEDS)

mpnn_r2 = {tt: r2_score(y_all[t_all==i], mpnn_oof[t_all==i]) for tt, i in task_map.items()}
for tt in TARGET_TYPES: log.metric(f'  MPNN [{tt}] R2={mpnn_r2[tt]:.4f}')
log.metric(f'>>> MPNN mean OOF R2 = {np.mean(list(mpnn_r2.values())):.4f}')

## 11. Polymer-invariance utilities

Kept independent of `USE_SSL` — translational invariance (RDKit random-SMILES round trip) and
oligomer/repetition invariance (joining repeat units at the two wildcard attachment points)
are needed for the invariance sanity check later regardless of whether the SSL branch runs.

In [ ]:
log.header('INVARIANCE UTILITIES')

def random_smiles_view(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return smi
    try: return Chem.MolToSmiles(m, doRandom=True)
    except Exception: return smi

def build_oligomer(smi, n_repeats=2):
    base = Chem.MolFromSmiles(smi)
    if base is None: return None
    if sum(1 for a in base.GetAtoms() if a.GetAtomicNum() == 0) != 2: return None
    combo, offset = None, 0
    bonds_to_add, dummies_to_remove = [], []
    prev_open_atom = prev_dummy2 = None
    for i in range(n_repeats):
        unit = Chem.Mol(base)
        combo = unit if combo is None else Chem.CombineMols(combo, unit)
        du = [a.GetIdx()+offset for a in unit.GetAtoms() if a.GetAtomicNum() == 0]
        if len(du) != 2: return None
        d1, d2 = du
        n1 = unit.GetAtomWithIdx(d1-offset).GetNeighbors()[0].GetIdx()+offset
        n2 = unit.GetAtomWithIdx(d2-offset).GetNeighbors()[0].GetIdx()+offset
        if i > 0:
            bonds_to_add.append((prev_open_atom, n1))
            dummies_to_remove += [prev_dummy2, d1]
        prev_open_atom, prev_dummy2 = n2, d2
        offset += unit.GetNumAtoms()
    rw = Chem.RWMol(combo)
    for a, b in bonds_to_add: rw.AddBond(a, b, Chem.BondType.SINGLE)
    for idx in sorted(set(dummies_to_remove), reverse=True): rw.RemoveAtom(idx)
    try:
        mol = rw.GetMol(); Chem.SanitizeMol(mol)
        if sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 0) != 2: return None
        return Chem.MolToSmiles(mol)
    except Exception:
        return None

_test_smi = train_df.smiles.dropna().sample(min(50, len(train_df)), random_state=SEED).tolist()
_ok = sum(1 for s in _test_smi if build_oligomer(s, 2) is not None)
log.info(f'oligomer builder: {_ok}/{len(_test_smi)} sampled train molecules produced a valid dimer')

## 12. Self-supervised branch (optional, gated by `USE_SSL`)

Condensed vs the notebook that produced 0.883 — same architecture, but structured as functions
so the whole branch is a no-op (returns zeros) when `USE_SSL = False`. Turn it back on only
after confirming the rest of v2's changes actually move LB toward 0.91; re-adding a component
that previously coincided with a score drop, on top of several other simultaneous changes, would
make it impossible to tell which change did what.

In [ ]:
log.header('SSL BRANCH (guarded by USE_SSL)')

SSL_MAX_HOURS, SSL_CHUNK, SSL_BATCH, SSL_MAX_LEN = 2.0, 200_000, 256, 128
SSL_DIM, SSL_HEADS, SSL_LAYERS, SSL_MASK_PROB, SSL_LR = 256, 8, 6, 0.15, 3e-4
SPECIALS = ['[PAD]', '[CLS]', '[MASK]', '[UNK]']
_base_chars = list("CNOFPSIBrclnos=#()-+[]@123456789/\\.%*{}~<>^ ")
VOCAB_LIST = SPECIALS + _base_chars
TOK2ID = {t: i for i, t in enumerate(VOCAB_LIST)}
PAD_ID, CLS_ID, MASK_ID, UNK_ID = TOK2ID['[PAD]'], TOK2ID['[CLS]'], TOK2ID['[MASK]'], TOK2ID['[UNK]']
SSL_VOCAB = len(VOCAB_LIST)

def ssl_tokenize(smi, max_len=SSL_MAX_LEN):
    ids = [CLS_ID] + [TOK2ID.get(c, UNK_ID) for c in smi[:max_len-1]]
    ids = ids[:max_len]
    mask = [1]*len(ids) + [0]*(max_len-len(ids))
    return ids + [PAD_ID]*(max_len-len(ids)), mask

class SSLEncoder(nn.Module):
    def __init__(s, vocab=SSL_VOCAB, dim=SSL_DIM, heads=SSL_HEADS, layers=SSL_LAYERS,
                 max_len=SSL_MAX_LEN, dropout=0.1):
        super().__init__()
        s.tok_emb = nn.Embedding(vocab, dim, padding_idx=PAD_ID)
        s.pos_emb = nn.Embedding(max_len, dim)
        enc_layer = nn.TransformerEncoderLayer(dim, heads, dim*4, dropout, activation='gelu', batch_first=True)
        s.encoder = nn.TransformerEncoder(enc_layer, layers)
        s.mlm_head = nn.Linear(dim, vocab); s.ln = nn.LayerNorm(dim)
    def encode(s, ids, attn_mask):
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0)
        h = s.encoder(s.tok_emb(ids) + s.pos_emb(pos), src_key_padding_mask=(attn_mask==0))
        return s.ln(h)
    def forward(s, ids, attn_mask): return s.mlm_head(s.encode(ids, attn_mask))
    def pool(s, ids, attn_mask):
        h = s.encode(ids, attn_mask); m = attn_mask.unsqueeze(-1).float()
        return (h*m).sum(1) / m.sum(1).clamp(min=1)

class SSLHead(nn.Module):
    def __init__(s, dim=SSL_DIM, n_tasks=7, hidden=256, dropout=0.2):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(dim, hidden), nn.SiLU(), nn.Dropout(dropout))
        s.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_tasks)])
    def forward(s, emb, t):
        z = s.net(emb); out = torch.zeros(emb.size(0), device=emb.device)
        for i, h in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = h(z[m]).squeeze(-1)
        return out

def run_ssl_pipeline():
    """Pretrain on the auxiliary corpus, then fine-tune per-fold heads on precomputed,
    invariance-averaged embeddings. Everything here executes once, in this run -- no
    checkpoint produced by an earlier run is ever loaded (rule 6.2.2/6.2.4)."""
    aux_path = f'{DATA_DIR}/smile_r3.csv'
    assert os.path.exists(aux_path), f'auxiliary file not found at {aux_path}'
    aux = pd.unique(pd.read_csv(aux_path, usecols=['smiles'], dtype=str)['smiles'].values)
    pool = pd.unique(np.concatenate([aux, train_df.smiles.values, test_df.smiles.values]))
    log.ok(f'pretraining pool: {len(pool):,} unique SMILES')

    model = SSLEncoder().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    t_start, step, seen = time.time(), 0, 0
    model.train()
    while (time.time()-t_start)/3600.0 < SSL_MAX_HOURS:
        idx = np.random.choice(len(pool), size=min(SSL_CHUNK, len(pool)), replace=False)
        for bstart in range(0, len(idx), SSL_BATCH):
            if (time.time()-t_start)/3600.0 >= SSL_MAX_HOURS: break
            chunk = pool[idx[bstart:bstart+SSL_BATCH]]
            toks = [ssl_tokenize(s) for s in chunk]
            ids = torch.LongTensor([t[0] for t in toks]).to(device)
            attn = torch.LongTensor([t[1] for t in toks]).to(device)
            labels = torch.full_like(ids, -100)
            eligible = (attn==1) & (ids!=CLS_ID) & (ids!=PAD_ID)
            prob = torch.rand(ids.shape, device=device)
            do_mask = eligible & (prob < SSL_MASK_PROB)
            labels[do_mask] = ids[do_mask]
            r = torch.rand(ids.shape, device=device)
            m_ids = ids.clone()
            m_ids[do_mask & (r<0.8)] = MASK_ID
            rr = do_mask & (r>=0.8) & (r<0.9)
            m_ids[rr] = torch.randint(len(SPECIALS), SSL_VOCAB, ids.shape, device=device)[rr]
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                loss = F.cross_entropy(model(m_ids, attn).view(-1, SSL_VOCAB), labels.view(-1), ignore_index=-100)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            step += 1; seen += len(chunk)
            if step % 200 == 0:
                log.metric(f'  SSL step {step} loss={loss.item():.4f} seen={seen:,} '
                           f'elapsed={(time.time()-t_start)/60:.1f}min')
    log.ok(f'SSL pretraining: {step} steps, {seen:,} seen, {seen/max(len(pool),1):.2f} passes')

    @torch.no_grad()
    def embed(smiles_list, bs=512):
        model.eval(); out = []
        for i in range(0, len(smiles_list), bs):
            toks = [ssl_tokenize(s) for s in smiles_list[i:i+bs]]
            ids = torch.LongTensor([t[0] for t in toks]).to(device)
            attn = torch.LongTensor([t[1] for t in toks]).to(device)
            out.append(model.pool(ids, attn).cpu().numpy())
        return np.concatenate(out, 0)

    def make_views(smi, n=3):
        views = [smi, random_smiles_view(smi)]
        d = build_oligomer(smi, 2)
        if d is not None: views.append(d)
        while len(views) < n: views.append(random_smiles_view(smi))
        return views[:n]

    train_views = [make_views(s) for s in smi_all]
    test_views = [make_views(s) for s in test_df.smiles.values]
    emb_tr = embed([v for vs in train_views for v in vs]).reshape(len(smi_all), 3, SSL_DIM)
    emb_te = embed([v for vs in test_views for v in vs]).reshape(len(test_df), 3, SSL_DIM)
    log.ok('embeddings precomputed for all folds')

    oof = np.zeros(len(y_all)); fold_models = []
    for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(smi_all)):
        torch.manual_seed(SEED+fold)
        tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
        for tt, i in task_map.items():
            ma, mb = (t_all[tr_i]==i), (t_all[va_i]==i)
            s = StandardScaler()
            if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
            if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
            tsc[i] = s
        head = SSLHead().to(device)
        opt2 = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
        best, best_state, pat = 1e18, None, 0
        et = torch.FloatTensor(emb_tr[tr_i]).to(device); ev = torch.FloatTensor(emb_tr[va_i,0]).to(device)
        tt_tr = torch.LongTensor(t_all[tr_i]).to(device); tt_va = torch.LongTensor(t_all[va_i]).to(device)
        y_tr = torch.FloatTensor(ya).to(device); y_va = torch.FloatTensor(yb).to(device)
        for ep in range(60):
            head.train(); perm = torch.randperm(len(tr_i))
            for bs in range(0, len(perm), 64):
                bidx = perm[bs:bs+64]
                eb = et[bidx].reshape(-1, SSL_DIM); tb = tt_tr[bidx].repeat_interleave(3)
                pred = head(eb, tb).view(len(bidx), 3)
                loss = F.huber_loss(pred[:,0], y_tr[bidx], delta=1.0) + 0.3*pred.var(dim=1).mean()
                opt2.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt2.step()
            head.eval()
            with torch.no_grad(): vl = F.mse_loss(head(ev, tt_va), y_va).item()
            if vl < best-1e-6: best, best_state, pat = vl, {k:v.cpu().clone() for k,v in head.state_dict().items()}, 0
            else:
                pat += 1
                if pat >= 12: break
        head.load_state_dict(best_state); head.eval()
        with torch.no_grad(): pn = head(ev, tt_va).cpu().numpy()
        pred = np.zeros_like(pn)
        for tt, i in task_map.items():
            m = (t_all[va_i]==i)
            if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
        oof[va_i] = pred; fold_models.append((tsc, head))
        log.metric(f'  SSL-head fold {fold+1}/{N_FOLDS} val_mse={best:.4f}')

    test_p = np.zeros(len(test_df))
    for tsc, head in fold_models:
        with torch.no_grad():
            e = torch.FloatTensor(emb_te.reshape(-1, SSL_DIM)).to(device)
            tb = torch.LongTensor(t_test).repeat_interleave(3).to(device)
            out = head(e, tb).cpu().numpy().reshape(len(test_df), 3).mean(1)
        q = np.zeros_like(out)
        for tt, i in task_map.items():
            m = (t_test==i)
            if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
        test_p += q / len(fold_models)
    return oof, test_p, fold_models, embed

if USE_SSL:
    ssl_oof, ssl_test_pred, ssl_fold_models, ssl_embed_fn = run_ssl_pipeline()
    ssl_r2 = {tt: r2_score(y_all[t_all==i], ssl_oof[t_all==i]) for tt, i in task_map.items()}
    log.metric(f'>>> SSL mean OOF R2 = {np.mean(list(ssl_r2.values())):.4f}')
else:
    ssl_oof, ssl_test_pred, ssl_fold_models, ssl_embed_fn = np.zeros(len(y_all)), np.zeros(len(test_df)), [], None
    log.info('USE_SSL=False -- SSL branch skipped, not included in the stack')

## 13. Test predictions (GBDT / NN / CNN / MPNN, + SSL if enabled)

In [ ]:
log.header('TEST PREDICTIONS')
test_pred = {}
for kind in ['lgbm', 'xgb', 'cb']:
    p = np.zeros(len(test_df))
    for tt in TARGET_TYPES:
        m = (test_df.target_type == tt).values
        Xte_full = drop_leaky(test_features[m], tt)
        preds = []
        for mm, cols in zip(tree_models[kind][tt], tree_cols[kind][tt]):
            Xte = Xte_full[cols] if cols is not None else Xte_full
            preds.append(mm.predict(Xte.values if kind == 'cb' else Xte))
        p[m] = np.column_stack(preds).mean(1)
    test_pred[kind] = p
    log.info(f'  {kind} done')

p = np.zeros(len(test_df))
for sc, tsc, model in nn_fold_models:
    Xs = np.nan_to_num(sc.transform(X_nn_test)).astype(np.float32)
    out = np.zeros(len(Xs))
    with torch.no_grad():
        for s0 in range(0, len(Xs), 512):
            e = min(s0+512, len(Xs))
            out[s0:e] = model(torch.FloatTensor(Xs[s0:e]).to(device),
                              torch.LongTensor(t_test[s0:e]).to(device)).cpu().numpy()
    q = np.zeros_like(out)
    for tt, i in task_map.items():
        m = (t_test == i)
        if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
    p += q / len(nn_fold_models)
test_pred['nn'] = p; log.info('  nn done')

p = np.zeros(len(test_df))
tds = SDS(test_df.smiles.values, np.zeros(len(test_df)), t_test)
for tsc, model in cnn_fold_models:
    pl = []
    with torch.no_grad():
        for xb, _, tb in DataLoader(tds, batch_size=512):
            pl.append(model(xb.to(device), tb.to(device)).cpu().numpy())
    out = np.concatenate(pl); q = np.zeros_like(out)
    for tt, i in task_map.items():
        m = (t_test == i)
        if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
    p += q / len(cnn_fold_models)
test_pred['cnn'] = p; log.info('  cnn done')

p = np.zeros(len(test_df))
gte = GraphDS(test_df.smiles.values, np.zeros(len(test_df)), t_test)
for tsc, model in mpnn_fold_models:
    pl = []
    model.eval()
    with torch.no_grad():
        for ai, ac, ef, am, adj, _, tb in DataLoader(gte, batch_size=256):
            ai,ac,ef,am,adj,tb = [x.to(device) for x in (ai,ac,ef,am,adj,tb)]
            pl.append(model(ai,ac,ef,am,adj,tb).cpu().numpy())
    out = np.concatenate(pl); q = np.zeros_like(out)
    for tt, i in task_map.items():
        m = (t_test == i)
        if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
    p += q / len(mpnn_fold_models)
test_pred['mpnn'] = p; log.info('  mpnn done')

if USE_SSL:
    test_pred['ssl'] = ssl_test_pred
log.ok(f'all base predictions generated: {sorted(test_pred)}')

## 14. Enhanced Ridge/LGBM stacking

Two changes from v1: (a) `mpnn` (and `ssl` if enabled) join the stack, plus each row's
**OOF standard deviation across base models** is added as an extra stacking feature — high
model disagreement is itself informative, not just noise to average away. (b) for each property
the meta-learner is chosen as **whichever of Ridge or a shallow LGBM scores higher in nested
CV** — Ridge assumes the base models' errors combine linearly, which is often wrong when one
model dominates on a given property.

In [ ]:
log.header('TRUE PARTNER FEATURES (reference for physics blend below)')
# _cmap / _truth / USES were built in section 4; kept here as a named reference only.
assert '_truth' in dir(), 'run section 4 (TRUE co-observed partner features) before this cell'
log.ok('partner table available')

In [ ]:
log.header('ENHANCED STACKING')
oof_dict = {'lgbm': tree_oof['lgbm'], 'xgb': tree_oof['xgb'], 'cb': tree_oof['cb'],
            'nn': nn_oof, 'cnn': cnn_oof, 'mpnn': mpnn_oof}
if USE_SSL:
    oof_dict['ssl'] = ssl_oof
names = sorted(oof_dict)
log.info(f'stacking inputs: {names}')

stack_r2, stack_meta, final = {}, {}, np.zeros(len(test_df))

for tt in TARGET_TYPES:
    m  = (train_df.target_type == tt).values
    mt = (test_df.target_type == tt).values
    base_tr = np.nan_to_num(np.column_stack([oof_dict[n][m] for n in names]))
    base_te = np.nan_to_num(np.column_stack([test_pred[n][mt] for n in names]))
    my = train_df.loc[m, 'target'].values

    std_tr = base_tr.std(axis=1, keepdims=True)
    std_te = base_te.std(axis=1, keepdims=True)
    mX = np.hstack([base_tr, std_tr])
    tX = np.hstack([base_te, std_te])

    # -- Ridge, alpha search
    best_a, best_ridge_s = 1.0, -1e18
    for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        sc_ = []
        for ti, vi in KFold(3, shuffle=True, random_state=SEED+200).split(mX):
            s = StandardScaler(); r = Ridge(alpha=a, random_state=SEED)
            r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
            sc_.append(r2_score(my[vi], r.predict(np.nan_to_num(s.transform(mX[vi])))))
        if np.mean(sc_) > best_ridge_s: best_ridge_s, best_a = np.mean(sc_), a

    # -- shallow LGBM meta-learner, same nested-CV protocol
    lgbm_meta_s = []
    for ti, vi in KFold(3, shuffle=True, random_state=SEED+200).split(mX):
        mdl = lgb.LGBMRegressor(n_estimators=200, max_depth=3, num_leaves=7,
                                learning_rate=0.05, min_child_samples=5,
                                random_state=SEED, verbose=-1)
        mdl.fit(mX[ti], my[ti])
        lgbm_meta_s.append(r2_score(my[vi], mdl.predict(mX[vi])))
    lgbm_meta_s = np.mean(lgbm_meta_s)

    use_lgbm_meta = lgbm_meta_s > best_ridge_s
    stack_meta[tt] = 'lgbm' if use_lgbm_meta else f'ridge(a={best_a:g})'

    oof_m = np.zeros(len(my))
    for ti, vi in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(mX):
        if use_lgbm_meta:
            mdl = lgb.LGBMRegressor(n_estimators=200, max_depth=3, num_leaves=7,
                                    learning_rate=0.05, min_child_samples=5,
                                    random_state=SEED, verbose=-1)
            mdl.fit(mX[ti], my[ti]); oof_m[vi] = mdl.predict(mX[vi])
        else:
            s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
            r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
            oof_m[vi] = r.predict(np.nan_to_num(s.transform(mX[vi])))
    stack_r2[tt] = r2_score(my, oof_m)

    if use_lgbm_meta:
        mdl = lgb.LGBMRegressor(n_estimators=200, max_depth=3, num_leaves=7,
                                learning_rate=0.05, min_child_samples=5,
                                random_state=SEED, verbose=-1)
        mdl.fit(mX, my); final[mt] = mdl.predict(tX)
    else:
        s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
        r.fit(np.nan_to_num(s.fit_transform(mX)), my)
        final[mt] = r.predict(np.nan_to_num(s.transform(tX)))

    log.metric(f'  [{tt}] meta={stack_meta[tt]:<12s} ridge_cv={best_ridge_s:.4f} '
               f'lgbm_cv={lgbm_meta_s:.4f} -> stack OOF R2={stack_r2[tt]:.4f}')

log.metric(f'>>> STACK MEAN OOF R2 = {np.mean(list(stack_r2.values())):.4f}')
hdr = ''.join(f'{n:>9}' for n in ['target']+names+['stack'])
print(chr(10)+hdr); print('-'*len(hdr))
for tt in TARGET_TYPES:
    row = [tree_r2['lgbm'][tt] if 'lgbm' in names else 0, tree_r2['xgb'][tt] if 'xgb' in names else 0,
           tree_r2['cb'][tt] if 'cb' in names else 0]
    per_name_r2 = {'lgbm': tree_r2['lgbm'][tt], 'xgb': tree_r2['xgb'][tt], 'cb': tree_r2['cb'][tt],
                   'nn': nn_r2[tt], 'cnn': cnn_r2[tt], 'mpnn': mpnn_r2[tt]}
    if USE_SSL: per_name_r2['ssl'] = ssl_r2[tt]
    print(f'{tt:<9}' + ''.join(f'{per_name_r2[n]:>9.4f}' for n in names) + f'{stack_r2[tt]:>9.4f}')

## 15. Physics blending — unified coalesce(true, predicted) version

v1 ran two disjoint passes (true-partner rows, then predicted-partner rows for the remainder).
v2 uses **whichever partner value is available for a given row** — true label if the partner
was measured, LGBM's prediction otherwise — in one pass, so every row that has *any* partner
information gets the physics estimate rather than only the subset with a true label. Per-property
shrinkage now comes from `SHRINK_OVERRIDE` (0.90 for `ei`/`eea`, where physics R2 is far above
the stack) instead of one blanket 0.75.

In [ ]:
PHYS = {
    'ei':  (['egc', 'eea'], lambda d: d[:, 0] + d[:, 1]),
    'eea': (['ei', 'egc'],  lambda d: d[:, 0] - d[:, 1]),
    'egb': (['egc'],        lambda d: d[:, 0]),
    'eps': (['nc'],         lambda d: d[:, 0] ** 2),
    'nc':  (['eps'],        lambda d: np.sqrt(np.clip(d[:, 0], 0, None))),
}

def _stack_oof_for(p):
    mtr = (train_df.target_type == p).values
    y = train_df.loc[mtr, 'target'].values
    mX = np.nan_to_num(np.column_stack([oof_dict[n][mtr] for n in names] +
                                        [np.column_stack([oof_dict[n][mtr] for n in names]).std(1)]))
    o = np.zeros(len(y))
    for a, b in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(mX):
        sc = StandardScaler(); r = Ridge(alpha=1.0, random_state=SEED)
        r.fit(np.nan_to_num(sc.fit_transform(mX[a])), y[a])
        o[b] = r.predict(np.nan_to_num(sc.transform(mX[b])))
    return mtr, y, o

def _coalesced_partner(df, props, pred_lookup):
    """True partner value where the label is known, else the LGBM prediction for that partner."""
    c = df['smiles'].map(_cmap)
    cols = []
    for q in props:
        true_v = c.map(_truth[q]).values.astype(np.float64)
        pred_v = pred_lookup[q]
        cols.append(np.where(np.isfinite(true_v), true_v, pred_v))
    return np.column_stack(cols)

log.header('PHYSICS BLEND -- COALESCED PARTNERS')
PRED_tr, PRED_te = {}, {}
for q in DFT_PROPS:
    PRED_tr[q] = np.mean([m.predict(drop_leaky(train_features, q)[c] if c is not None
                                    else drop_leaky(train_features, q))
                          for m, c in zip(tree_models['lgbm'][q], tree_cols['lgbm'][q])], axis=0)
    PRED_te[q] = np.mean([m.predict(drop_leaky(test_features, q)[c] if c is not None
                                    else drop_leaky(test_features, q))
                          for m, c in zip(tree_models['lgbm'][q], tree_cols['lgbm'][q])], axis=0)
log.info('partner predictions ready (used as fallback where true label is unknown)')

applied = 0
for p, (srcs, fn) in PHYS.items():
    mtr, y, stack_oof = _stack_oof_for(p)
    mte = (test_df.target_type == p).values
    pred_lookup_tr = {q: PRED_tr[q][mtr] for q in srcs}
    D_tr = _coalesced_partner(train_df[mtr], srcs, pred_lookup_tr)
    est = fn(D_tr)
    if not np.isfinite(est).all():
        log.warn(f'  {p}: non-finite physics estimate on {(~np.isfinite(est)).sum()} rows -- skipped')
        continue
    cal = np.zeros(len(est))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(est):
        A = np.c_[est[a], np.ones(len(a))]
        w_, *_ = np.linalg.lstsq(A, y[a], rcond=None)
        cal[b] = np.c_[est[b], np.ones(len(b))] @ w_
    bw, br = 0.0, -1e18
    for w in np.arange(0, 1.001, 0.05):
        r = r2_score(y, (1-w)*stack_oof + w*cal)
        if r > br: br, bw = r, w
    shrink = SHRINK_OVERRIDE.get(p, SHRINK_DEFAULT)
    w_use = shrink * bw
    if w_use <= 0:
        log.info(f'  {p}: physics adds nothing (w=0) -- untouched'); continue
    A = np.c_[est, np.ones(len(est))]
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    pred_lookup_te = {q: PRED_te[q][mte] for q in srcs}
    D_te = _coalesced_partner(test_df[mte], srcs, pred_lookup_te)
    est_te = fn(D_te)
    okte = np.isfinite(est_te)
    cal_te = np.c_[est_te[okte], np.ones(okte.sum())] @ coef
    idx = np.where(mte)[0][okte]
    final[idx] = (1-w_use)*final[idx] + w_use*cal_te
    applied += len(idx)
    log.metric(f'  {p:4s} n={mtr.sum():<4d} | stack {r2_score(y, stack_oof):.4f} phys {r2_score(y, cal):.4f} '
               f'blend {br:.4f} | shrink={shrink} w={bw:.2f}->{w_use:.2f} | test rows touched {okte.sum()}')
log.ok(f'coalesced physics applied to {applied} test rows total')
assert np.isfinite(final).all(), 'physics blending produced non-finite values' 

## 16. Pseudo-labeling (optional, gated by `PSEUDO_LABEL`, off by default)

Only touches the small-n properties, and only rows where the base models already agree tightly
(`std < PSEUDO_CONF_STD` in z-scored units). This is the riskiest change in v2 — a systematic
bias shared by all base models would get reinforced, not corrected — so validate everything
else against LB first, and only then flip this on and compare.

In [ ]:
if PSEUDO_LABEL:
    log.header('PSEUDO-LABELING (small-n properties)')
    pseudo_rows = []
    for tt in TARGET_TYPES:
        mtr = (train_df.target_type == tt).values
        if mtr.sum() >= SMALL:
            continue
        mte = (test_df.target_type == tt).values
        base_te = np.column_stack([test_pred[n][mte] for n in names])
        agree = base_te.std(1)
        conf = agree < np.quantile(agree, 0.25)  # tightest quartile of agreement
        conf &= (agree < PSEUDO_CONF_STD * (train_df.loc[mtr,'target'].std()))
        n_conf = conf.sum()
        if n_conf < PSEUDO_MIN_N:
            log.info(f'  {tt}: only {n_conf} confident test rows -- skipped'); continue
        idx = np.where(mte)[0][conf]
        pdf = pd.DataFrame({'smiles': test_df.iloc[idx].smiles.values,
                            'target_type': tt, 'target': final[idx]})
        pseudo_rows.append(pdf)
        log.metric(f'  {tt}: pseudo-labeled {n_conf} test rows')
    if pseudo_rows:
        pseudo_df = pd.concat(pseudo_rows, ignore_index=True)
        log.ok(f'{len(pseudo_df)} pseudo-labeled rows generated -- '
               f'append to train_df and re-run sections 7 (GBDT) for affected target_types '
               f'if you want to use these; not auto-applied here so you can inspect them first')
    else:
        log.info('no properties met the pseudo-labeling confidence bar')
else:
    log.info('PSEUDO_LABEL=False -- skipped')

## 17. Explainability (SHAP, unchanged approach from v1)

In [ ]:
log.header('EXPLAINABILITY')

def explain_prediction_report(target_type, top_k=8):
    mask = (train_df.target_type == target_type).values
    X_full = drop_leaky(train_features[mask].reset_index(drop=True), target_type)
    cols0 = tree_cols['lgbm'][target_type][0]
    X = X_full[cols0] if cols0 is not None else X_full
    model = tree_models['lgbm'][target_type][0]
    ex = shap.TreeExplainer(model)
    sv = ex.shap_values(X.sample(min(50, len(X)), random_state=SEED))
    imp = pd.Series(np.abs(sv).mean(0), index=X.columns).sort_values(ascending=False).head(top_k)
    log.info(f'[{target_type}] top {top_k} features by mean |SHAP|:')
    for c, v in imp.items():
        kind = 'partner/physics' if c in USES else 'structural'
        log.info(f'    {c:30s} {kind:12s} {v:.4f}')
    return imp

for tt in TARGET_TYPES:
    explain_prediction_report(tt)

## 18. Invariance sanity check (adapts to whichever SMILES-only model is active)

In [ ]:
log.header('INVARIANCE SANITY CHECK')
_sample_idx = np.random.RandomState(SEED).choice(len(test_df), size=min(100, len(test_df)), replace=False)
_deltas = []

if USE_SSL and ssl_fold_models:
    _tsc0, _head0 = ssl_fold_models[0]
    for i in _sample_idx:
        row = test_df.iloc[i]; v1 = row.smiles; v2 = random_smiles_view(v1)
        if v2 == v1: continue
        e1 = ssl_embed_fn([v1]); e2 = ssl_embed_fn([v2])
        t_ = torch.LongTensor([task_map[row.target_type]]).to(device)
        with torch.no_grad():
            p1 = _head0(torch.FloatTensor(e1).to(device), t_).item()
            p2 = _head0(torch.FloatTensor(e2).to(device), t_).item()
        _deltas.append(abs(p1-p2))
    log.info('measured on the SSL head')
else:
    _tsc0, _model0 = cnn_fold_models[0]
    for i in _sample_idx:
        row = test_df.iloc[i]; v1 = row.smiles; v2 = random_smiles_view(v1)
        if v2 == v1: continue
        t_ = torch.LongTensor([task_map[row.target_type]]).to(device)
        with torch.no_grad():
            p1 = _model0(torch.LongTensor([tok(v1)]).to(device), t_).item()
            p2 = _model0(torch.LongTensor([tok(v2)]).to(device), t_).item()
        _deltas.append(abs(p1-p2))
    log.info('measured on the CNN (USE_SSL=False)')

if _deltas:
    log.metric(f'mean |pred(canonical) - pred(random view)| on {len(_deltas)} test molecules: '
               f'{np.mean(_deltas):.4f} (z-scored units; smaller = more invariant)')

## 19. Submission + final compliance self-audit

In [ ]:
log.header('SUBMISSION')
for tt in TARGET_TYPES:
    m = (test_df.target_type == tt).values
    v = train_df.loc[train_df.target_type == tt, 'target']
    lo, hi = v.min(), v.max(); pad = 0.05*(hi-lo)
    final[m] = np.clip(final[m], lo-pad, hi+pad)

bad = ~np.isfinite(final)
if bad.any():
    log.warn(f'{bad.sum()} non-finite predictions -> LightGBM fallback')
    final[bad] = test_pred['lgbm'][bad]

sub = pd.DataFrame({'id': test_df.id.values, 'target': final})
assert len(sub) == len(test_df)
assert sub.target.notna().all() and np.isfinite(sub.target.values).all()
assert sub.id.nunique() == len(sub)
sub.to_csv(os.path.join(WORK_DIR, 'submission.csv'), index=False)
log.ok(f'submission.csv written {sub.shape}')
print(sub.groupby(test_df.target_type).target.agg(['min','mean','max']).round(3))

In [ ]:
log.header('FINAL COMPLIANCE SELF-AUDIT')
_other_inputs = [p for p in glob.glob('/kaggle/input/*') if os.path.abspath(p) != os.path.abspath(DATA_DIR)]
if _other_inputs:
    log.warn(f'other datasets attached: {_other_inputs} -- confirm none were read above')
else:
    log.ok('no other attached datasets detected under /kaggle/input')
log.info(f'SEED={SEED} fixed throughout; USE_SSL={USE_SSL}; PSEUDO_LABEL={PSEUDO_LABEL}')
_total_min = (time.time() - T_NOTEBOOK_START) / 60
log.metric(f'total notebook wall time: {_total_min:.1f} min ({_total_min/60:.2f} h) -- '
           f'check against your actual Kaggle session limit')